# Video Scene Analysis — Knowledge Graph RAG + Chain-of-Thought Reasoning Demo

This notebook demonstrates the end-to-end pipeline for analysing dashcam / CCTV
footage using **Gemini 2.5 Pro** via **LiteLLM**. Unlike the crash-only variant,
this version instructs the model to **describe every visible object, road user,
environmental feature, and event** in each frame — not just collisions. It
extends how to reinforce the causality analysis with a **knowledge graph-backed
Retrieval-Augmented Generation (RAG) pipeline**.

The core problem this addresses: vanilla multimodal LLMs make systematic
errors when analysing collision footage — particularly **temporal attribution
errors** (blaming a later-arriving vehicle for an earlier incident) and
**missing hit-and-run scenarios** (failing to identify the striking vehicle
because it left the scene). The enhanced notebook mitigated these with
hand-crafted causal reasoning prompts, but those prompts were specific to one
video. This notebook generalises the approach by encoding domain expertise
into a **structured knowledge graph** that is retrieved at inference time
based on scene metadata.

**Enhancement**
**Collision Pattern Knowledge Graph** — a NetworkX property graph encoding
5 collision archetypes (HGV roundabout sideswipe, rear-end stationary,
hit-and-run, failure to yield, lane change sideswipe), 6 vehicle type
nodes, 4 road geometry nodes, 4 UK traffic regulations, and typed edges
(`INVOLVES`, `OCCURS_ON`, `APPLIES_TO`, `OFTEN_FOLLOWED_BY`).
- **Scored graph traversal retrieval** — patterns are ranked by matching
detected vehicle types, road geometry, and keyword hints extracted from
the scene, replacing the hardcoded prompt additions.
- **Camera auto-detection** — optical flow stability analysis on the first
5 seconds classifies the camera as static CCTV, forward dashcam, or
handheld, and injects the appropriate ego-motion context into the prompt.
- **Composable prompt architecture** — camera context, retrieved collision
patterns (with their `ERRORS TO AVOID` warnings), applicable UK
regulations, and a chain-of-thought instruction structure are assembled
programmatically before the output schema.
- **Perception-to-cognition chain-of-thought** — the model is required to
populate a `reasoning_trace` (global perception → vehicle inventory →
interaction analysis → causal conclusion) *before* the structured output,
forcing explicit reasoning rather than pattern-matching.
- **KG-grounded verification pass** — Pass 2 cross-checks the initial
analysis against the top-ranked pattern's `common_misidentifications`
list, directly targeting the error classes most relevant to the scene.
- **GraphML export** — the knowledge graph is exportable to Neo4j or Azure
Cosmos DB Gremlin for production deployment.

**Pipeline**
```
Video file
│
├─▶ CV Pre-processing (key frame extraction, camera auto-detection)
│
├─▶ KG Pattern Retrieval (vehicle types + road geometry → scored patterns)
│
├─▶ Prompt Composition (camera context + RAG patterns + CoT structure)
│
├─▶ LLM Pass 1 — KG-augmented analysis with reasoning trace
│
├─▶ LLM Pass 2 — KG-grounded verification against common errors
│
└─▶ Pydantic Output (VideoSceneAnalysisOutput + ReasoningTrace)
```


**Pipeline steps:**
1. Setup & configuration
2. Import / define Pydantic schemas (with `ReasoningTrace` extension)
3. Build the traffic collision knowledge graph
4. Define KG retrieval functions
5. Camera auto-detection via optical flow
6. Load video & CV pre-processing (key frame extraction)
7. Retrieve relevant patterns from the knowledge graph
8. Compose the KG-augmented prompt
9. Build payload & call Gemini 2.5 Pro (Pass 1 — CoT analysis)
10. KG-grounded verification (Pass 2)
11. Parse the verified response into Pydantic models
12. Explore and export results

**Prerequisites:**
- `opencv-python-headless` (key frame extraction, optical flow)
- `networkx` (knowledge graph)
- `litellm` (LLM gateway to Gemini 2.5 Pro)
- `pydantic` (structured output schemas)
- A valid LiteLLM API key and access to the Zurich LiteLLM proxy

## 1. Setup & Configuration

In [ ]:
import litellm
import json
import base64
import logging
import cv2
import sys
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from dataclasses import dataclass, field
from enum import Enum
from pydantic import BaseModel, Field
from collections.abc import Sequence
from pathlib import Path
from IPython.display import display, JSON, Video, Image as IPImage

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
# ── LiteLLM credentials ─────────────────────────────────────────────
# Replace with your own credentials or load from environment variables.

LITELLM_API_KEY = "sk-jijtUNjqiqtTr1DSQUeKMQ"
LITELLM_BASE_URL = "https://genai-lounge-nx-litellm-uat-emea.zurich.com"

litellm.api_key = LITELLM_API_KEY
litellm.api_base = LITELLM_BASE_URL
litellm.suppress_debug_info = True

# Model identifier — routed through LiteLLM to the Gemini 2.5 Pro deployment
MODEL_NAME = "gemini/gemini-2.5-pro"

print(f"LiteLLM configured — base: {LITELLM_BASE_URL}")
print(f"Model: {MODEL_NAME}")

## 2. Pydantic Schemas for Scene Description (with Reasoning Trace)

In [ ]:
# Ensure the project root is on sys.path for local imports
sys.path.insert(0, str(Path(".").resolve().parent))

class ReasoningTrace(BaseModel):
    global_perception: str = Field(..., description="Scene-level observations")
    vehicle_inventory: str = Field(..., description="All vehicles with IDs and descriptions")
    interaction_analysis: str = Field(..., description="Vehicle interaction analysis")
    causal_conclusion: str = Field(..., description="Final causal attribution with evidence")


from video_analysis_poc.video_llm.schema.video import (
    VideoObjectObservation,
    CrashInvolvedParty,
    WitnessedCrashDetail,
    SceneEvent,
    EnvironmentContext,
    VideoSceneAnalysisOutput,
)
print("Imported scene-description schemas from video_analysis_poc.video_llm.schema.video")

## 3. Traffic Collision Knowledge Graph

The knowlefge Graph (KG) encodes collision patterns, vehicle types, road geometries, causal chains and common misidentification pitfalls using graph library (NetworkX) with a property-graph model.

In [ ]:
@dataclass
class CollisionPattern:
    pattern_id: str
    pattern_name: str
    description: str
    visual_signatures: list[str]
    causal_chain: str
    common_misidentifications: list[str]
    vehicle_types: list[str]
    road_types: list[str]
    keywords: list[str]
    severity_range: str = "low to medium"
    example_narrative: str = ""
    
@dataclass
class TrafficRegulation:
    reg_id: str
    jurisdiction: str
    category: str
    rule_text: str
    analysis_implication: str

print("Dataclasses defined.")

In [ ]:
# Load knowledge graph data
_KG_DATA = json.loads('''{"vehicle_types": [{"id": "car", "label": "Car / SUV", "swept_path_risk": "low"}, {"id": "truck", "label": "Truck / HGV / Lorry", "swept_path_risk": "high"}, {"id": "van", "label": "Van / LCV", "swept_path_risk": "medium"}, {"id": "bus", "label": "Bus / Coach", "swept_path_risk": "high"}, {"id": "motorcycle", "label": "Motorcycle", "swept_path_risk": "low"}, {"id": "pedestrian", "label": "Pedestrian", "swept_path_risk": "none"}], "road_types": [{"id": "roundabout", "multi_lane": true, "turning_required": true}, {"id": "motorway", "multi_lane": true, "turning_required": false}, {"id": "intersection", "multi_lane": true, "turning_required": true}, {"id": "urban_road", "multi_lane": false, "turning_required": false}], "patterns": [{"pattern_id": "roundabout_sideswipe_hgv", "pattern_name": "HGV Sideswipe on Roundabout", "description": "A large vehicle (truck/HGV) enters a roundabout and its swept turning path encroaches on an adjacent smaller vehicle, causing a side scrape.", "visual_signatures": ["Large vehicle and smaller vehicle entering roundabout side-by-side", "Smaller vehicle suddenly stops on roundabout after truck passes", "Truck continues without stopping (scrape-and-leave)", "No rear-end impact visible - damage is lateral", "Gap between vehicles narrows as they navigate the curve"], "causal_chain": "HGV enters outer lane -> turning arc exceeds lane width -> trailer encroaches inner lane -> contact -> car stops, HGV continues", "common_misidentifications": ["WRONG: Vehicle described as inexplicably stopped (the stop was CAUSED by the sideswipe)", "WRONG: Later-arriving vehicle blamed for collision (they merely navigated around)", "WRONG: Impact classified as rear-end when actually lateral", "WRONG: Truck not identified as involved because it left the scene"], "vehicle_types": ["truck", "car"], "road_types": ["roundabout"], "keywords": ["sideswipe", "swept path", "roundabout", "truck", "HGV", "scrape"], "example_narrative": "A green Yodel truck and black SUV approach a multi-lane roundabout. Truck in outer lane, SUV in inner lane. Both enter at ~15s. Between 17-19s the truck rear swings into SUV lane. SUV stops at ~20s after being scraped. Truck continues. Later vehicles (van ~25s, red SUV ~38s) just navigate around the stopped SUV."}, {"pattern_id": "rear_end_stationary", "pattern_name": "Rear-End Collision with Stationary Vehicle", "description": "A moving vehicle collides with the rear of a stationary vehicle. Classic failure-to-observe.", "visual_signatures": ["Clear closing speed between moving and stationary vehicle", "Impact as sudden deceleration at point of contact", "Both vehicles in same lane before impact", "Post-impact: both stationary, striking vehicle directly behind"], "causal_chain": "Lead vehicle stationary -> following vehicle fails to observe -> closing speed -> rear-end impact", "common_misidentifications": ["WRONG: Vehicle passing a stopped car confused with rear-end impact", "WRONG: Vehicle slowing near stopped car assumed to have hit it"], "vehicle_types": ["car", "truck", "van"], "road_types": ["roundabout", "motorway", "urban_road"], "keywords": ["rear-end", "stationary", "following distance", "braking"]}, {"pattern_id": "hit_and_run_scrape", "pattern_name": "Hit-and-Run / Scrape-and-Leave", "description": "A vehicle makes contact and continues driving without stopping. Common with large vehicles.", "visual_signatures": ["One vehicle stops after contact; the other continues", "Departing vehicle shows no behaviour change", "Victim vehicle stops in unusual position mid-lane"], "causal_chain": "Contact -> victim stops -> striking vehicle continues -> scene appears as if victim stopped for no reason", "common_misidentifications": ["WRONG: Victim described as inexplicably stopped because striking vehicle left frame", "WRONG: Later-arriving vehicle blamed because actual culprit is gone"], "vehicle_types": ["truck", "car", "van", "bus"], "road_types": ["roundabout", "motorway", "urban_road", "intersection"], "keywords": ["hit and run", "scrape and leave", "departed scene"]}, {"pattern_id": "roundabout_failure_yield", "pattern_name": "Failure to Yield at Roundabout Entry", "description": "Vehicle enters roundabout without giving way to circulating traffic.", "visual_signatures": ["Vehicle enters without slowing while another circulates", "Near-miss or impact at entry point", "Circulating vehicle brakes hard or swerves"], "causal_chain": "Vehicle A circulating -> Vehicle B enters without yielding -> paths converge -> impact or avoidance", "common_misidentifications": ["WRONG: Normal give-way waiting confused with near-miss"], "vehicle_types": ["car", "van", "truck"], "road_types": ["roundabout"], "keywords": ["give way", "yield", "roundabout entry", "side impact"]}, {"pattern_id": "lane_change_sideswipe", "pattern_name": "Lane Change Sideswipe", "description": "Vehicle changes lane without checking blind spot, contacting adjacent vehicle.", "visual_signatures": ["Vehicle drifts laterally into occupied lane", "Side-to-side contact, same direction of travel", "No significant speed differential"], "causal_chain": "Vehicle A in lane 1 -> lane change into lane 2 -> Vehicle B in lane 2 -> lateral contact", "common_misidentifications": ["WRONG: Misclassified as head-on when both moving same direction"], "vehicle_types": ["car", "van", "truck"], "road_types": ["motorway", "urban_road"], "keywords": ["lane change", "blind spot", "sideswipe", "merge"]}], "regulations": [{"reg_id": "uk_roundabout_priority", "jurisdiction": "UK", "category": "roundabout", "rule_text": "Give way to traffic from the right when entering a roundabout (Highway Code Rule 185).", "analysis_implication": "Vehicles on the roundabout have priority. Entering vehicles must yield."}, {"reg_id": "uk_lane_discipline", "jurisdiction": "UK", "category": "roundabout", "rule_text": "Use left lane for left/straight; right lane for right/full circle (Highway Code Rule 186).", "analysis_implication": "Lane position at entry indicates intended exit."}, {"reg_id": "uk_hgv_swept_path", "jurisdiction": "UK", "category": "general", "rule_text": "Large vehicles may need more than one lane when turning (Highway Code Rule 221).", "analysis_implication": "HGVs encroaching into adjacent lanes is a known sideswipe risk."}, {"reg_id": "uk_hit_and_run", "jurisdiction": "UK", "category": "general", "rule_text": "Failing to stop and report an accident is an offence under Road Traffic Act 1988 s.170.", "analysis_implication": "If a vehicle departs after contact, flag as potential offence."}]}''')

G = nx.DiGraph()

# Vehicle type nodes
for vt in _KG_DATA["vehicle_types"]:
    G.add_node(f"vtype:{vt['id']}", node_type="vehicle_type", **vt)
    
# Road geometry nodes
for rt in _KG_DATA["road_types"]:
    G.add_node(f"road:{rt['id']}", node_type="road_geometry", **rt)
    
# Collision pattern nodes and edges
for p in _KG_DATA["patterns"]:
    pid = p["pattern_id"]
    G.add_node(f"pattern:{pid}", node_type="collision_pattern", **p)
    for vt in p["vehicle_types"]:
        G.add_edge(f"pattern:{pid}", f"vtype:{vt}", relation="INVOLVES")
    for rt in p["road_types"]:
        G.add_edge(f"pattern:{pid}", f"road:{rt}", relation="OCCURS_ON")

# Causal edge: HGV sideswipes often become hit-and-runs
G.add_edge("pattern:roundabout_sideswipe_hgv", "pattern:hit_and_run_scrape",
           relation="OFTEN_FOLLOWED_BY")

# Regulation nodes
for r in _KG_DATA["regulations"]:
    G.add_node(f"reg:{r['reg_id']}", node_type="regulation", **r)
    
G.add_edge("reg:uk_roundabout_priority", "road:roundabout", relation="APPLIES_TO")
G.add_edge("reg:uk_lane_discipline", "road:roundabout", relation="APPLIES_TO")
G.add_edge("reg:uk_hgv_swept_path", "road:roundabout", relation="APPLIES_TO")

print(f"Knowledge graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Patterns: {len(_KG_DATA['patterns'])}, Regulations: {len(_KG_DATA['regulations'])}")

## 4. Knowledge Graph Retrieval Functions

In [ ]:
def retrieve_patterns(graph, detected_vehicle_types, road_type=None, keyword_hints=None):
    results = []
    for node_id, attrs in graph.nodes(data=True):
        if attrs.get("node_type") != "collision_pattern":
            continue
        score = 0
        neighbors = [n for n in graph.successors(node_id)
                     if graph.edges[node_id, n].get("relation") == "INVOLVES"]
        matched = [n.split(":")[1] for n in neighbors if n.split(":")[1] in detected_vehicle_types]
        score += len(matched) * 2
        road_neighbors = [n for n in graph.successors(node_id)
                         if graph.edges[node_id, n].get("relation") == "OCCURS_ON"]
        if road_type and any(road_type in n for n in road_neighbors):
            score += 3
        if keyword_hints:
            score += len(set(keyword_hints) & set(attrs.get("keywords", [])))
        if score > 0:
            results.append({**attrs, "relevance_score": score})
    results.sort(key=lambda x: x["relevance_score"], reverse=True)
    return results

def retrieve_regulations(graph, road_type=None, jurisdiction="UK"):
    results = []
    for node_id, attrs in graph.nodes(data=True):
        if attrs.get("node_type") != "regulation" or attrs.get("jurisdiction") != jurisdiction:
            continue
        applies_to = [n for n in graph.successors(node_id)
                     if graph.edges[node_id, n].get("relation") == "APPLIES_TO"]
        if (road_type and any(road_type in n for n in applies_to)) or attrs.get("category") == "general":
            results.append(attrs)
    return results

print("Retrieval functions defined.")

## 5. Camera Auto-Detection

In [ ]:
class CameraMotionType(str, Enum):
    STATIC = "static"
    EGO_FORWARD = "ego_forward"
    HANDHELD = "handheld"
    
@dataclass
class CameraProfile:
    motion_type: CameraMotionType
    avg_stability: float
    prompt_context: str

def detect_camera_profile(video_path: Path, sample_seconds: int = 5) -> CameraProfile:
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    ret, prev_frame = cap.read()
    if not ret:
        cap.release()
        return CameraProfile(CameraMotionType.EGO_FORWARD, 0.5, "Assume dashcam.")
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    flow_mags: list[float] = []
    for _ in range(min(int(sample_seconds * fps), 150)):
        ret, frame = cap.read()
        if not ret: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, np.zeros((*prev_gray.shape, 2), dtype=np.float32), 0.5, 3, 15, 3, 5, 1.2, 0)
        mag = float(np.mean(np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)))
        flow_mags.append(mag)
        prev_gray = gray
    cap.release()
    
    if not flow_mags:
        return CameraProfile(CameraMotionType.EGO_FORWARD, 0.5, "Insufficient frames.")
    
    avg_flow = float(np.mean(flow_mags))
    stability = 1.0 / (1.0 + avg_flow)
    flow_std = float(np.std(flow_mags))
    
    if stability > 0.85:
        return CameraProfile(CameraMotionType.STATIC, stability,
            "Fixed CCTV. No ego-motion. All motion = real vehicle movement.")
    elif flow_std / (avg_flow + 1e-6) > 1.5:
        return CameraProfile(CameraMotionType.HANDHELD, stability,
            "Handheld/body-worn. Significant jitter. Distinguish vehicle behaviour from camera shake.")
    else:
        return CameraProfile(CameraMotionType.EGO_FORWARD, stability,
            "Front-facing dashcam on moving vehicle. ALL background motion = ego-vehicle movement. "
            "Focus on RELATIVE motion between vehicles.")

print("Camera detection defined.")

## 6. Load Video & Strucutred CV Pre-Processing

Supplementing the full video with still images at critical moments gives the model better spatial resolution exactly when it matters — particularly around the moments when vehicles interact closely (e.g. a truck and car entering a roundabout side-by-side).

In [ ]:
# ── Point this to your local dashcam / CCTV sample ─────────────────
VIDEO_PATH = Path("C:\\Users\\AMIR.RAHIMI\\Downloads\\Dashcam\\Inbound_CHO_Dashcam_20-50-780086.mp4")
CAMERA_TYPE = "dashcam_front"
SUPPORTED_FORMATS = {"mp4", "avi", "mov", "mkv", "webm"}
VIDEO_MIME_TYPES = {"mp4": "video/mp4", "avi": "video/x-msvideo", "mov": "video/quicktime",
                    "mkv": "video/x-matroska", "webm": "video/webm"}

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"
size_mb = VIDEO_PATH.stat().st_size / (1024 * 1024)
print(f"Video: {VIDEO_PATH.name} ({size_mb:.2f} MB)")

camera_profile = detect_camera_profile(VIDEO_PATH)
print(f"Camera: {camera_profile.motion_type.value} (stability: {camera_profile.avg_stability:.3f})")


In [ ]:
def extract_key_frames(video_path, timestamps, max_dim=768):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    for ts in timestamps:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(ts * fps))
        ret, frame = cap.read()
        if not ret: continue
        h, w = frame.shape[:2]
        scale = min(max_dim / w, max_dim / h, 1.0)
        if scale < 1.0:
            frame = cv2.resize(frame, (int(w * scale), int(h * scale)))
        ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
        if not ok: continue
        
        frames.append((ts, base64.b64encode(buf.tobytes()).decode("utf-8")))
    cap.release()
    return frames

KEY_TIMESTAMPS = [0, 5, 10, 14, 15, 16, 17, 18, 19, 20, 21, 22, 25, 30, 35, 38, 39, 40, 42, 45, 50]
key_frames = extract_key_frames(VIDEO_PATH, KEY_TIMESTAMPS)
print(f"Extracted {len(key_frames)} key frames")

## 7. Retrieve Relevant Patterns from Knowledge Graph

In [ ]:
DETECTED_VEHICLE_TYPES = ["truck", "car", "van"]
DETECTED_ROAD_TYPE = "roundabout"
KEYWORD_HINTS = ["sideswipe", "roundabout", "stopped", "truck"]

retrieved_patterns = retrieve_patterns(G, DETECTED_VEHICLE_TYPES, DETECTED_ROAD_TYPE, KEYWORD_HINTS)
retrieved_regulations = retrieve_regulations(G, DETECTED_ROAD_TYPE, "UK")

print(f"Retrieved {len(retrieved_patterns)} patterns:")
for p in retrieved_patterns:
    print(f"  [{p['relevance_score']}] {p['pattern_name']}")
print(f"\nRetrieved {len(retrieved_regulations)} regulations:")
for r in retrieved_regulations:
    print(f"  - {r['reg_id']}")

## 8. Compose Knowledge Graph-Augmented Prompt

In [ ]:
def format_rag_context(patterns, regulations):
    lines = ["DOMAIN EXPERTISE FROM KNOWLEDGE BASE:", ""]
    for p in patterns[:3]:
        lines.append(f"### Pattern: {p['pattern_name']}")
        lines.append(f"Description: {p['description']}")
        sigs = '; '.join(p.get('visual_signatures', []))
        lines.append(f"Visual signatures: {sigs}")
        lines.append(f"Causal chain: {p['causal_chain']}")
        misids = '; '.join(p.get('common_misidentifications', []))
        lines.append(f"ERRORS TO AVOID: {misids}")
        narr = p.get('example_narrative', '')
        if narr:
            lines.append(f"Example: {narr}")
        lines.append("")
    if regulations:
        lines.append("APPLICABLE REGULATIONS:")
        for r in regulations:
            lines.append(f"- {r['rule_text']}")
            lines.append(f"  Implication: {r['analysis_implication']}")
    return "\n".join(lines)

rag_context = format_rag_context(retrieved_patterns, retrieved_regulations)
print(f"RAG context: {len(rag_context)} chars")
print(rag_context[:600])

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert multimodal video analyst for road scene understanding. "
    "Provide comprehensive structured descriptions of dashcam/CCTV footage.\n\n"
    f"CAMERA: {camera_profile.prompt_context}\n\n"
    "PRINCIPLES:\n"
    "- Establish CAUSAL relationships. Trace why vehicles stop.\n"
    "- Large vehicles have wide turning arcs causing sideswipes on roundabouts.\n"
    "- Striking vehicle may leave (hit-and-run). Trace timeline backwards.\n"
    "- Do NOT confuse later-arriving vehicles with the one that CAUSED the event.\n"
    "- Use the domain expertise patterns and still frames provided.\n\n"
    "REASONING CHAIN:\n"
    "1. GLOBAL PERCEPTION: Overall scene\n"
    "2. VEHICLE INVENTORY: Every vehicle with ID\n"
    "3. INTERACTION ANALYSIS: Trajectory relationships\n"
    "4. CAUSAL CONCLUSION: Event classification with evidence"
)
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

In [ ]:
TASK_PROMPT = rag_context + '''

---

Analyse the video and still frames using the domain patterns above.
Follow the perception-to-cognition chain. Return JSON:

{
  "reasoning_trace": {
    "global_perception": "",
    "vehicle_inventory": "",
    "interaction_analysis": "",
    "causal_conclusion": ""
  },
  "environment": {"weather": "...", "lighting": "...", "road_type": "...",
                  "road_surface": "...", "traffic_density": "...", "additional_notes": "..."},
  "detected_objects": [{"object_type": "...", "description": "...",
    "first_seen_seconds": 0.0, "last_seen_seconds": 0.0,
    "position_in_frame": "...", "confidence": 0.0}],
  "events": [{"event_type": "...", "timestamp_seconds": 0.0,
    "timestamp_display": "HH:MM:SS", "description": "...",
    "involved_objects": ["..."], "severity": "...", "confidence": 0.0,
    "witnessed_crash_detail": {
      "impact_type": "...",
      "involved_parties": [{"road_user_type": "...", "description": "...",
        "action_before_event": "...", "action_after_event": "...",
        "apparent_damage": "..."}],
      "probable_cause": "...", "fault_indicators": ["..."],
      "sequence_of_events": "...", "post_crash_actions": "...",
      "estimated_speed_range": "...", "dashcam_vehicle_involved": false}}],
  "summary": ""
}

Instructions:
- FIRST populate reasoning_trace, THEN structured output.
- Check conclusions against ERRORS TO AVOID from knowledge base.
- For crash events, ALWAYS populate witnessed_crash_detail.
- For non-crash events, set witnessed_crash_detail to null.
- If striking vehicle left scene, note explicitly.
- Return ONLY JSON.
'''
print(f"Task prompt: {len(TASK_PROMPT)} chars")

## 9. Build Payload & Call LLM (Pass 1)

In [ ]:
logger.info("Encoding video...")
with open(VIDEO_PATH, "rb") as f:
    video_b64 = base64.b64encode(f.read()).decode("utf-8")
video_mime = VIDEO_MIME_TYPES.get(VIDEO_PATH.suffix.lstrip(".").lower(), "video/mp4")

content_parts = [
    {"type": "text", "text": f"[{CAMERA_TYPE}] Video: {VIDEO_PATH.name}"},
    {"type": "image_url", "image_url": {"url": f"data:{video_mime};base64,{video_b64}"}},
]
for ts, b64 in key_frames:
    content_parts.append({"type": "text", "text": f"[FRAME {ts:.0f}s] Check vehicle positions."})
    content_parts.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}})
content_parts.append({"type": "text", "text": TASK_PROMPT})

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": content_parts}]
print(f"Payload: {len(content_parts)} parts (1 video + {len(key_frames)} frames)")

In [ ]:
import time as _time

def extract_json(resp):
    content = resp.choices[0].message.content
    if not content: raise ValueError("Empty")
    c = content.strip()
    if c.startswith("```"): c = c.split("\n", 1)[1]
    if c.endswith("```"): c = c[:-3].strip()
    return json.loads(c)

logger.info("Pass 1: KG-augmented analysis...")
_t0 = _time.perf_counter()
raw_p1 = await litellm.acompletion(model=MODEL_NAME, messages=messages,
    temperature=0.0, response_format={"type": "json_object"}, cache={"no-store": True})
logger.info("Pass 1 done in %.1fs", _time.perf_counter() - _t0)

parsed_p1 = extract_json(raw_p1)
print(f"Keys: {list(parsed_p1.keys())}")

if "reasoning_trace" in parsed_p1:
    rt = parsed_p1["reasoning_trace"]
    for step in ["global_perception", "vehicle_inventory", "interaction_analysis", "causal_conclusion"]:
        print(f"\n-- {step.upper()} --")
        print(rt.get(step, "N/A")[:400])

p1_crashes = [e for e in parsed_p1.get("events", []) if e.get("event_type") in ("collision", "near_miss")]
print(f"\nPass 1: {len(p1_crashes)} crash/near-miss events")
for c in p1_crashes:
    print(f"  [{c['event_type']}] {c.get('timestamp_display','?')}: {c.get('description','')[:120]}")

## 10. Knowledge Graph-Grounded Verification (Pass 2)

In [ ]:
top_pat = retrieved_patterns[0] if retrieved_patterns else None
misid_check = ""
if top_pat:
    misids = "; ".join(top_pat.get("common_misidentifications", []))
    misid_check = f"\nKB CHECK: Top pattern is '{top_pat['pattern_name']}'. Common errors: {misids}\nVerify you did NOT make these errors.\n"

VERIFY = f"""You previously produced:
{json.dumps(parsed_p1, indent=2)}
{misid_check}
Verify: 1) Causality of stopped vehicles 2) Temporal ordering 3) Truck swept-path at 15-22s
4) Whether later vehicles (38-40s) actually made contact or just passed by
Provide CORRECTED JSON, same schema. Return ONLY JSON."""

verify_parts = content_parts[:-1] + [{"type": "text", "text": VERIFY}]
msgs_p2 = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": verify_parts}]

logger.info("Pass 2: Verification...")
_t0 = _time.perf_counter()
raw_p2 = await litellm.acompletion(model=MODEL_NAME, messages=msgs_p2,
    temperature=0.0, response_format={"type": "json_object"}, cache={"no-store": True})
logger.info("Pass 2 done in %.1fs", _time.perf_counter() - _t0)

parsed = extract_json(raw_p2)
p2_crashes = [e for e in parsed.get("events", []) if e.get("event_type") in ("collision", "near_miss")]
print(f"Pass 2: {len(p2_crashes)} crash/near-miss")
for c in p2_crashes:
    print(f"  [{c['event_type']}] {c.get('timestamp_display','?')}: {c.get('description','')[:120]}")

## 11. Parse into Pydantic Models

In [ ]:
reasoning_trace = ReasoningTrace(**parsed["reasoning_trace"]) if "reasoning_trace" in parsed else None
environment = EnvironmentContext(**parsed["environment"]) if parsed.get("environment") else None
detected_objects = [VideoObjectObservation(**o) for o in parsed.get("detected_objects", [])]

events = []
for evt_raw in parsed.get("events", []):
    evt = dict(evt_raw)
    cd_raw = evt.pop("witnessed_crash_detail", None)
    cd = None
    if cd_raw and isinstance(cd_raw, dict):
        dd = dict(cd_raw)
        parties = [CrashInvolvedParty(**p) for p in dd.pop("involved_parties", [])]
        cd = WitnessedCrashDetail(involved_parties=parties, **dd)
    events.append(SceneEvent(witnessed_crash_detail=cd, **evt))

output = VideoSceneAnalysisOutput(
    task_name="video_scene_analysis_kg_rag", camera_type=CAMERA_TYPE,
    video_file_name=VIDEO_PATH.name, environment=environment,
    detected_objects=detected_objects, events=events,
    summary=parsed.get("summary"), raw_output=parsed)

n_crash = sum(1 for e in events if e.event_type in ("collision", "near_miss"))
print(f"Objects: {len(detected_objects)}, Events: {len(events)} ({n_crash} crash/near-miss)")
print(f"Summary: {output.summary}")

## 12. Explore the Results

In [ ]:

if reasoning_trace:
    print("=" * 60, "\n  CHAIN-OF-THOUGHT REASONING TRACE\n", "=" * 60)
    for s in ["global_perception", "vehicle_inventory", "interaction_analysis", "causal_conclusion"]:
        print(f"\n-- {s.replace('_',' ').upper()} --")
        print(getattr(reasoning_trace, s))

In [ ]:
if output.environment:
    e = output.environment
    print("=" * 60, "\n  Environment\n", "=" * 60)
    for f in ["weather","lighting","road_type","road_surface","traffic_density","additional_notes"]:
        v = getattr(e, f, None)
        if v: print(f"  {f:<18s} {v}")

In [ ]:
if output.detected_objects:
    df = pd.DataFrame([o.model_dump() for o in output.detected_objects])
    display(df.style.format({"confidence": "{:.2%}"}).set_caption(f"Objects ({len(df)})"))

In [ ]:

for i, ev in enumerate(output.events, 1):
    print(f"\n{'='*60}\n  Event {i}: {ev.event_type}\n{'='*60}")
    print(f"  Time: {ev.timestamp_display} ({ev.timestamp_seconds:.1f}s)")
    print(f"  Severity: {ev.severity or 'N/A'}, Confidence: {ev.confidence:.2%}")
    print(f"  Description: {ev.description}")
    if ev.witnessed_crash_detail:
        cd = ev.witnessed_crash_detail
        print(f"  Impact: {cd.impact_type}, Cause: {cd.probable_cause}")
        print(f"  Sequence: {cd.sequence_of_events}")
        for j, p in enumerate(cd.involved_parties, 1):
            print(f"    Party {j}: {p.road_user_type} - {p.description}")
            print(f"      Before: {p.action_before_event}")
            print(f"      After: {p.action_after_event}")

In [ ]:
for label, resp in [("Pass 1", raw_p1), ("Pass 2", raw_p2)]:
    u = getattr(resp, "usage", None)
    if u: print(f"{label}: prompt={getattr(u,'prompt_tokens','?')}, "
                 f"completion={getattr(u,'completion_tokens','?')}, total={getattr(u,'total_tokens','?')}")

In [ ]:
out_path = Path("./scene_analysis_logic_rag_results.json")
out_path.write_text(json.dumps(output.model_dump(), indent=2, default=str))
print(f"Results: {out_path.resolve()}")

# Export KG as GraphML
G_exp = G.copy()
for n, a in G_exp.nodes(data=True):
    for k, v in list(a.items()):
        if isinstance(v, (list, dict)): a[k] = json.dumps(v)
        elif isinstance(v, bool): a[k] = str(v)
nx.write_graphml(G_exp, "./collision_knowledge_graph.graphml")
print("KG exported: ./collision_knowledge_graph.graphml")

## 13. Visualization

In [ ]:
# Load the exported GraphML (or just use G directly)
G_viz = G.copy()

# Define colour and shape mapping by node type
_NODE_COLORS = {
    "collision_pattern": "#E74C3C",  # red
    "vehicle_type": "#3498DB",  # blue
    "road_geometry": "#2ECC71",  # green
    "regulation": "#F39C12",  # amber
}
_EDGE_COLORS = {
    "INVOLVES": "#888888",
    "OCCURS_ON": "#2ECC71",
    "APPLIES_TO": "#F39C12",
    "OFTEN_FOLLOWED_BY": "#E74C3C",
}

# Build layout — shell layout groups nodes by type for readability
shells = [
    [n for n, d in G_viz.nodes(data=True) if d.get("node_type") == "collision_pattern"],
    [
        n
        for n, d in G_viz.nodes(data=True)
        if d.get("node_type") in ("vehicle_type", "road_geometry")
    ],
    [n for n, d in G_viz.nodes(data=True) if d.get("node_type") == "regulation"],
]

shells = [s for s in shells if s]  # drop empty shells
pos = nx.shell_layout(G_viz, nlist=shells, rotate=0.2)

# Node properties
node_colors = [
    _NODE_COLORS.get(G_viz.nodes[n].get("node_type", ""), "#CCCCCC")
    for n in G_viz.nodes()
]
node_sizes = [
    2200 if G_viz.nodes[n].get("node_type") == "collision_pattern" else 1400
    for n in G_viz.nodes()
]

# Short labels: strip prefix (e.g. "pattern:", "vtype:")
labels = {}
for n, d in G_viz.nodes(data=True):
    if d.get("node_type") == "collision_pattern":
        labels[n] = d.get("pattern_name", n.split(":")[-1])
    elif d.get("node_type") == "vehicle_type":
        labels[n] = d.get("label", n.split(":")[-1])
    elif d.get("node_type") == "regulation":
        labels[n] = (
            d.get("reg_id", n.split(":")[-1])
            .replace("uk_", "")
            .replace("_", " ")
            .title()
        )
    else:
        labels[n] = n.split(":")[-1].replace("_", " ").title()
        
# Edge colours
edge_colors = [
    _EDGE_COLORS.get(G_viz.edges[e].get("relation", ""), "#AAAAAA")
    for e in G_viz.edges()
]

# Draw
fig, ax = plt.subplots(1, 1, figsize=(18, 12), facecolor="white")

nx.draw_networkx_nodes(
    G_viz,
    pos,
    ax=ax,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.9,
    edgecolors="white",
    linewidths=1.5,
)

nx.draw_networkx_labels(
    G_viz, pos, labels=labels, ax=ax, font_size=7, font_weight="bold"
)

nx.draw_networkx_edges(
    G_viz,
    pos,
    ax=ax,
    edge_color=edge_colors,
    arrows=True,
    arrowsize=14,
    width=1.5,
    alpha=0.7,
    connectionstyle="arc3,rad=0.08",
)

# Edge relation labels
edge_labels = {(u, v): d.get("relation", "") for u, v, d in G_viz.edges(data=True)}
nx.draw_networkx_edge_labels(
    G_viz,
    pos,
    edge_labels=edge_labels,
    ax=ax,
    font_size=5.5,
    font_color="#555555",
    rotate=False,
)

# Legend
legend_handles = [
    mpatches.Patch(color=c, label=t.replace("_", " ").title())
    for t, c in _NODE_COLORS.items()
]
legend_handles += [
    Line2D([0], [0], color=c, linewidth=2, label=r.replace("_", " ").title())
    for r, c in _EDGE_COLORS.items()
]
ax.legend(
    handles=legend_handles,
    loc="upper left",
    fontsize=8,
    title="Node / Edge Types",
    title_fontsize=9,
    framealpha=0.9,
)

ax.set_title(
    "Traffic Collision Knowledge Graph\n"
    f"({G_viz.number_of_nodes()} nodes, {G_viz.number_of_edges()} edges)",
    fontsize=14,
    fontweight="bold",
    pad=16,
)
ax.axis("off")
plt.tight_layout()
plt.savefig(
    "./collision_kg_visualisation.png", dpi=150, bbox_inches="tight", facecolor="white"
)
plt.show()

print(f"Visualisation saved: ./collision_kg_visualisation.png")
